# Bank Marketing Campaign Optimization

**Publication-upgrade companion | Python, SQL, classification, calibration, targeting**

Source: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), CC BY 4.0. The notebook calls the authoritative `analysis.py` pipeline and does not maintain a competing implementation.

## Context & Methods

The empirical question is whether response models improve resource-constrained targeting when restricted to decision-time information. The primary specification uses customer and prior-campaign history available before campaign-list construction. Training-only cross-validation selects among a dummy, class-weighted logistic regression, random forest, and histogram gradient boosting. Nested training-only predictions select calibration. A fixed development holdout supports prespecified uncertainty, budget, information-set, duplicate, and leakage comparisons.

### Key Assumptions

- Rows are contact records; customer IDs are unavailable.
- Month and weekday do not recover complete row-level dates.
- The holdout was reported by earlier repository versions and is not pristine external validation.
- Subscription propensity and observed ranking efficiency are not causal treatment effects.

In [1]:
from pathlib import Path
from IPython.display import Markdown, display
import pandas as pd

from analysis import run_analysis

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
PROJECT_DIR = Path.cwd()
results = run_analysis(PROJECT_DIR)
summary = results["summary"]

In [2]:
display(Markdown(f"""## tl;dr

The conservative **{summary['primary_information_set']}** model selected **{summary['selected_base_model']}** with **{summary['calibration_method_label']} calibration**. On the fixed {summary['holdout_rows']:,}-row development holdout it achieved **{summary['pr_auc']:.3f} PR-AUC**, **{summary['roc_auc']:.3f} ROC-AUC**, and **{summary['brier_score']:.3f} Brier score**. The top 20% captured **{summary['top_20_capture']:.1%}** of observed subscribers at **{summary['top_20_lift']:.2f}x lift**. Adding post-contact `duration` raised PR-AUC to **{summary['leaky_model_pr_auc']:.3f}**, illustrating invalid leakage rather than deployable value.
"""))

## tl;dr

The conservative **Strict planning (primary)** model selected **Random Forest** with **isotonic calibration**. On the fixed 8,236-row development holdout it achieved **0.339 PR-AUC**, **0.706 ROC-AUC**, and **0.087 Brier score**. The top 20% captured **45.5%** of observed subscribers at **2.27x lift**. Adding post-contact `duration` raised PR-AUC to **0.651**, illustrating invalid leakage rather than deployable value.


## Data and decision-time audit

In [3]:
display(results["data_quality"])
display(results["feature_availability_audit"][["Feature", "Planning_Time_Available", "Pre_Contact_Available", "Post_Contact_Only", "Primary_Model_Allowed", "Rationale"]])

,Check,Value,Status,Analytical_Risk
0,Source hash,74adfc578bf77a7ff4bb1ba4a9f8709d9e3c6907342959...,Pass,Wrong source version
1,Raw rows,41188,Pass,Unexpected extract size
2,Exact duplicate rows,12,Sensitivity analyzed,Unknown record identity
3,Analysis rows,41176,Pass,Row reconciliation
4,Missing cells,0,Pass,Silent missingness
5,Literal unknown cells,12716,Documented category,Unknown not missing at random
6,Rows containing unknown,10698,Documented category,Coverage
7,Positive outcomes,4639,Pass,Class balance
8,campaign_prior minimum,0,Pass,Invalid transformation
9,campaign_prior maximum,55,Documented,Contact-frequency tail


,Feature,Planning_Time_Available,Pre_Contact_Available,Post_Contact_Only,Primary_Model_Allowed,Rationale
0,age,True,True,False,True,Bank-held client attribute.
1,job,True,True,False,True,Bank-held client attribute; unknown remains a ...
2,marital,True,True,False,True,Bank-held client attribute; divorced includes ...
3,education,True,True,False,True,Bank-held client attribute; unknown remains a ...
4,default,True,True,False,True,Bank-held credit attribute; unknown remains a ...
5,housing,True,True,False,True,Bank-held credit attribute; unknown remains a ...
6,loan,True,True,False,True,Bank-held credit attribute; unknown remains a ...
7,contact,False,True,False,False,Known for a scheduled contact but not necessar...
8,month,False,True,False,False,Current-contact timing; operational sensitivit...
9,day_of_week,False,True,False,False,Current-contact timing; operational sensitivit...


## Results

In [4]:
display(results["cv_summary"])
display(results["holdout_metrics"])
display(results["bootstrap_intervals"])
display(results["budget_metrics"].head(4))
display(results["information_set_comparison"])
display(results["duplicate_sensitivity"])
display(results["leakage_audit"])
display(results["permutation_importance"].head(10))

,Model,ROC_AUC_Mean,ROC_AUC_Std,PR_AUC_Mean,PR_AUC_Std,Brier_Score_Mean,Log_Loss_Mean,Selection_Rank,Selected
0,Random Forest,0.7038,0.0105,0.3530,0.0153,0.1871,0.5668,1,True
1,Histogram Gradient Boosting,0.7049,0.0096,0.3530,0.0179,0.1938,0.5829,2,False
2,Logistic Regression,0.6960,0.0119,0.3374,0.0197,0.2019,0.6004,3,False
3,Dummy Baseline,0.5000,0.0000,0.1127,0.0001,0.1000,0.3520,4,False


,Model,Role,Selected_Base_Model,ROC_AUC,PR_AUC,Brier_Score,Log_Loss,Top20_Capture,Top20_Lift
0,Dummy Baseline,Locked benchmark,False,0.5000,0.1127,0.1000,0.3521,0.1940,0.9694
1,Logistic Regression,Locked benchmark,False,0.7030,0.3268,0.2025,0.6023,0.4429,2.2134
2,Random Forest,Locked benchmark,True,0.7070,0.3411,0.1885,0.5708,0.4634,2.3157
3,Histogram Gradient Boosting,Locked benchmark,False,0.7070,0.3421,0.1946,0.5858,0.4483,2.2403
4,Random Forest (isotonic calibration),Final selected model,True,0.7062,0.3391,0.0873,0.3100,0.4547,2.2726


,Metric,Estimate,CI95_Lower,CI95_Upper,Bootstrap_Samples
0,ROC_AUC,0.7062,0.6868,0.7245,1000
1,PR_AUC,0.3391,0.3073,0.3720,1000
2,Brier_Score,0.0873,0.0823,0.0917,1000
3,Top10_Capture,0.3427,0.3160,0.3708,1000
4,Top20_Capture,0.4547,0.4261,0.4826,1000
5,Top30_Capture,0.5582,0.5279,0.5851,1000
6,Top20_Lift,2.2726,2.1295,2.4119,1000


,contact_share,selected_record_share,contact_records,subscribers_captured,conversion_rate,lift,responder_capture,contacts_per_observed_subscriber,random_expected_subscribers,random_conversion_rate,random_lift,random_responder_capture,random_contacts_per_observed_subscriber
0,0.1000,0.1000,824,318,0.3859,3.4251,0.3427,2.5912,92.8451,0.1127,1.0000,0.1000,8.8750
1,0.2000,0.2001,1648,422,0.2561,2.2726,0.4547,3.9052,185.6901,0.1127,1.0000,0.2001,8.8750
2,0.3000,0.3000,2471,518,0.2096,1.8605,0.5582,4.7703,278.4225,0.1127,1.0000,0.3000,8.8750
3,0.4000,0.4001,3295,595,0.1806,1.6026,0.6412,5.5378,371.2676,0.1127,1.0000,0.4001,8.8750


,Information_Set,Primary,Feature_Count,Features,CV_PR_AUC,CV_ROC_AUC,Holdout_PR_AUC,Holdout_ROC_AUC,Holdout_Brier_Score,Top20_Capture,Top20_Lift
0,Strict planning (primary),True,10,age|job|marital|education|default|housing|loan...,0.3530,0.7038,0.3391,0.7062,0.0873,0.4547,2.2726
1,Planning plus macro context,False,15,age|job|marital|education|default|housing|loan...,0.4471,0.7912,0.4544,0.8055,0.0770,0.6530,3.2635
2,Operational pre-contact,False,14,age|job|marital|education|default|housing|loan...,0.4225,0.7743,0.4227,0.7810,0.0804,0.6013,3.0050
3,Operational plus macro context,False,19,age|job|marital|education|default|housing|loan...,0.4673,0.7996,0.4845,0.8133,0.0751,0.6509,3.2527


,Specification,Rows,ROC_AUC,PR_AUC,Brier_Score,Log_Loss,Top20_Capture,Top20_Lift
0,Exact duplicates removed (primary),41176,0.7062,0.3391,0.0873,0.3100,0.4547,2.2726
1,All source rows retained,41188,0.7023,0.3488,0.0861,0.3077,0.4526,2.2624


,Feature_Set,Deployable,Includes_Duration,ROC_AUC,PR_AUC,Brier_Score,Log_Loss,Top20_Capture,Top20_Lift
0,Operational plus macro context,True,False,0.8133,0.4845,0.0751,0.2652,0.6509,3.2527
1,Invalid: operational plus macro plus duration,False,True,0.9463,0.6515,0.0568,0.1786,0.8750,4.3729


,Feature,Importance_Mean,Importance_SD,Scoring,Interpretation
7,pdays,0.0409,0.0044,PR-AUC decrease,"Predictive association, not causal effect"
0,age,0.0297,0.0026,PR-AUC decrease,"Predictive association, not causal effect"
9,poutcome,0.0280,0.0035,PR-AUC decrease,"Predictive association, not causal effect"
4,default,0.0173,0.0050,PR-AUC decrease,"Predictive association, not causal effect"
8,previous,0.0144,0.0027,PR-AUC decrease,"Predictive association, not causal effect"
1,job,0.0144,0.0029,PR-AUC decrease,"Predictive association, not causal effect"
3,education,0.0072,0.0025,PR-AUC decrease,"Predictive association, not causal effect"
2,marital,0.0060,0.0016,PR-AUC decrease,"Predictive association, not causal effect"
5,housing,-0.0008,0.0013,PR-AUC decrease,"Predictive association, not causal effect"
6,loan,-0.0013,0.0017,PR-AUC decrease,"Predictive association, not causal effect"


In [5]:
display(Markdown(f"""## Takeaways

- **Information timing dominates interpretation.** The strict planning model is the defensible primary analysis; richer operational and macro sets answer different decisions.
- **Uncertainty is visible.** Final PR-AUC is {summary['pr_auc']:.3f} (95% bootstrap CI {summary['pr_auc_ci95'][0]:.3f}-{summary['pr_auc_ci95'][1]:.3f}); model differences use paired resampling.
- **Budget value is observational.** The top 20% captured {summary['top_20_capture']:.1%} of observed subscribers, not customers caused to subscribe.
- **The strongest-looking model is invalid.** Post-contact duration raises PR-AUC to {summary['leaky_model_pr_auc']:.3f} but cannot be used at the targeting decision.
- **Next study:** prospective temporal validation followed by randomized policy evaluation with customer IDs, cost, and value outcomes.
"""))

## Takeaways

- **Information timing dominates interpretation.** The strict planning model is the defensible primary analysis; richer operational and macro sets answer different decisions.
- **Uncertainty is visible.** Final PR-AUC is 0.339 (95% bootstrap CI 0.307-0.372); model differences use paired resampling.
- **Budget value is observational.** The top 20% captured 45.5% of observed subscribers, not customers caused to subscribe.
- **The strongest-looking model is invalid.** Post-contact duration raises PR-AUC to 0.651 but cannot be used at the targeting decision.
- **Next study:** prospective temporal validation followed by randomized policy evaluation with customer IDs, cost, and value outcomes.
